# Week 2 — PDF Chunking Pipeline (3 Strategies)\n",
    "Extracts and chunks all Knowledge Base PDFs. Text is extracted **once**, then 3 chunking strategies are applied.\n",
    "\n",
    "| Strategy | Chunk Size | Overlap | Best For |\n",
    "|----------|-----------|---------|----------|\n",
    "| **S1: Sliding-512** | 512 words | 128 words | General baseline |\n",
    "| **S2: Sliding-256** | 256 words | 64 words | Re-ranking precision (Week 3) |\n",
    "| **S3: Paragraph-aware** | paragraph boundary | merge if < 80w | Regulatory/structured docs |\n",
    "\n",
    "**Outputs:**\n",
    "- `Dataset/chunks/chunks_512w.csv`\n",
    "- `Dataset/chunks/chunks_256w.csv`\n",
    "- `Dataset/chunks/chunks_paragraph.csv`\n",
    "\n",
    "Best strategy selected in **Week 3** after cross-encoder re-ranking evaluation."

## Cell 0 — Repo Root & Config

In [ ]:
import sys
import os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / "config.py").exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import (
    REGULATORY_PATH,
    CONSUMER_PROTECTION_PATH,
    PAYMENT_INDUSTRY_PATH,
    COMPLAINT_PROCEDURES_PATH,
)

CHUNKS_DIR    = os.path.join("Dataset", "chunks")
RESULTS_DIR   = os.path.join("pdf_chunking", "Results")
os.makedirs(CHUNKS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

OUT_512W      = os.path.join(CHUNKS_DIR, "chunks_512w.csv")
OUT_256W      = os.path.join(CHUNKS_DIR, "chunks_256w.csv")
OUT_PARAGRAPH = os.path.join(CHUNKS_DIR, "chunks_paragraph.csv")
CHUNK_STATS   = os.path.join(RESULTS_DIR, "chunking_stats.json")

PDF_SOURCES = {
    "Regulatory"         : REGULATORY_PATH,
    "Consumer_Protection": CONSUMER_PROTECTION_PATH,
    "Payment_Industry"   : PAYMENT_INDUSTRY_PATH,
    "Synthetic_Policies" : COMPLAINT_PROCEDURES_PATH,
}

print(f"Repo root : {_root}")
print()
for cat, path in PDF_SOURCES.items():
    pdfs = list(Path(path).glob("*.pdf")) if os.path.exists(path) else []
    print(f"  [{len(pdfs):>2} PDFs]  {cat}")

## Cell 1 — Install & Imports

In [ ]:
import sys
!{sys.executable} -m pip install -q pdfplumber

In [ ]:
import re
import json
import pdfplumber
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("All imports OK")

## Cell 2 — Helper Functions (Extraction + 3 Chunkers)

In [ ]:
# ── Extraction ───────────────────────────────────────────────────────────────

def extract_text_from_pdf(pdf_path):
    """Extract full text from a PDF. Returns a single cleaned string."""
    text_parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)
    except Exception as e:
        print(f"    WARNING: could not read {pdf_path.name} — {e}")
        return ""
    return "\n".join(text_parts)


def clean_text(text):
    """Remove excessive whitespace and non-ASCII noise."""
    text = re.sub(r'\n{3,}',   '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ',   text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()


# ── Strategy 1: Sliding Window 512w ──────────────────────────────────────────

def chunk_sliding(text, chunk_size=512, overlap=128):
    """
    Fixed sliding window on word tokens.
    stride = chunk_size - overlap
    """
    words  = text.split()
    step   = chunk_size - overlap
    chunks = []
    idx    = 0
    start  = 0
    while start < len(words):
        end        = min(start + chunk_size, len(words))
        chunk_text = " ".join(words[start:end]).strip()
        if len(chunk_text) > 50:
            chunks.append((idx, chunk_text))
            idx += 1
        start += step
        if end == len(words):
            break
    return chunks


# ── Strategy 2: Sliding Window 256w ──────────────────────────────────────────

def chunk_sliding_256(text):
    """Smaller chunks: 256 words, 64 overlap. Better for re-ranking precision."""
    return chunk_sliding(text, chunk_size=256, overlap=64)


# ── Strategy 3: Paragraph-Aware ──────────────────────────────────────────────

def chunk_paragraph(text, max_words=512, min_words=80):
    """
    Respect natural document structure:
    1. Split on double newlines (paragraph boundaries)
    2. Merge short paragraphs (< min_words) with next paragraph
    3. Sub-chunk any paragraph that exceeds max_words using sliding window

    This preserves section boundaries in regulatory and policy documents.
    """
    # Step 1: split into raw paragraphs
    raw_paras = [p.strip() for p in re.split(r'\n{2,}', text) if p.strip()]

    # Step 2: merge short paragraphs into the next one
    merged = []
    buffer = ""
    for para in raw_paras:
        if buffer:
            combined = buffer + " " + para
            if len(buffer.split()) < min_words:
                buffer = combined
                continue
            else:
                merged.append(buffer)
                buffer = para
        else:
            buffer = para
    if buffer:
        merged.append(buffer)

    # Step 3: sub-chunk any paragraph that's too large
    final_chunks = []
    idx = 0
    for para in merged:
        word_count = len(para.split())
        if word_count > max_words:
            # sub-chunk with sliding window, no overlap (clean paragraph boundaries)
            sub_chunks = chunk_sliding(para, chunk_size=max_words, overlap=0)
            for _, sub_text in sub_chunks:
                if len(sub_text) > 50:
                    final_chunks.append((idx, sub_text))
                    idx += 1
        else:
            if len(para) > 50:
                final_chunks.append((idx, para))
                idx += 1
    return final_chunks


print("Helper functions defined.")
print("  S1: Sliding-512  (512w chunks, 128w overlap)")
print("  S2: Sliding-256  (256w chunks,  64w overlap)")
print("  S3: Paragraph    (natural boundaries, max 512w, merge < 80w)")

## Cell 3 — Extract Text From All PDFs (Once)
Text extraction is the slow step. We do it **once** and store results in memory,  
then apply all 3 chunking strategies without re-reading the files.

In [ ]:
# extracted_docs: list of {source_file, category, text}
extracted_docs = []

for category, folder_path in PDF_SOURCES.items():
    folder   = Path(folder_path)
    pdf_list = sorted(folder.glob("*.pdf"))

    if not pdf_list:
        print(f"  [{category}] No PDFs found — skipping")
        continue

    print(f"Extracting [{category}]  ({len(pdf_list)} PDFs) ...")
    for pdf_path in tqdm(pdf_list, desc=f"  {category}"):
        raw   = extract_text_from_pdf(pdf_path)
        clean = clean_text(raw)
        if clean:
            extracted_docs.append({
                "source_file": pdf_path.name,
                "category"   : category,
                "text"       : clean,
                "word_count" : len(clean.split()),
            })

print(f"\nExtracted {len(extracted_docs)} documents")
print(f"Total words: {sum(d['word_count'] for d in extracted_docs):,}")

## Cell 4 — Apply All 3 Chunking Strategies

In [ ]:
def apply_chunker(docs, chunker_fn, strategy_name):
    """Apply a chunking function to all extracted docs. Returns a list of chunk dicts."""
    chunks   = []
    chunk_id = 0
    for doc in docs:
        doc_chunks = chunker_fn(doc["text"])
        for chunk_idx, chunk_text in doc_chunks:
            chunks.append({
                "chunk_id"   : f"{strategy_name}_{chunk_id:06d}",
                "source_file": doc["source_file"],
                "category"   : doc["category"],
                "chunk_index": chunk_idx,
                "text"       : chunk_text,
                "word_count" : len(chunk_text.split()),
            })
            chunk_id += 1
    return chunks


print("Applying chunking strategies...")

chunks_512w      = apply_chunker(extracted_docs, lambda t: chunk_sliding(t, 512, 128), "s1")
print(f"  S1 Sliding-512   : {len(chunks_512w):,} chunks")

chunks_256w      = apply_chunker(extracted_docs, chunk_sliding_256,                   "s2")
print(f"  S2 Sliding-256   : {len(chunks_256w):,} chunks")

chunks_paragraph = apply_chunker(extracted_docs, chunk_paragraph,                     "s3")
print(f"  S3 Paragraph     : {len(chunks_paragraph):,} chunks")

## Cell 5 — Save All 3 CSVs

In [ ]:
df_512w      = pd.DataFrame(chunks_512w)
df_256w      = pd.DataFrame(chunks_256w)
df_paragraph = pd.DataFrame(chunks_paragraph)

df_512w.to_csv(OUT_512W,      index=False)
df_256w.to_csv(OUT_256W,      index=False)
df_paragraph.to_csv(OUT_PARAGRAPH, index=False)

print(f"Saved: chunks_512w.csv        ({len(df_512w):,} rows,  {os.path.getsize(OUT_512W)/1024/1024:.1f} MB)")
print(f"Saved: chunks_256w.csv        ({len(df_256w):,} rows,  {os.path.getsize(OUT_256W)/1024/1024:.1f} MB)")
print(f"Saved: chunks_paragraph.csv   ({len(df_paragraph):,} rows,  {os.path.getsize(OUT_PARAGRAPH)/1024/1024:.1f} MB)")

## Cell 6 — Side-by-Side Comparison

In [ ]:
strategies = {
    "S1: Sliding-512" : df_512w,
    "S2: Sliding-256" : df_256w,
    "S3: Paragraph"   : df_paragraph,
}

print("\n" + "="*72)
print("          CHUNKING STRATEGIES — SIDE BY SIDE COMPARISON")
print("="*72)
print(f"  {'Strategy':<20} {'Chunks':>8} {'Min W':>7} {'Max W':>7} {'Avg W':>7} {'Median W':>9}")
print("-"*72)
for name, df in strategies.items():
    wc = df["word_count"]
    print(f"  {name:<20} {len(df):>8,} {wc.min():>7} {wc.max():>7} {wc.mean():>7.0f} {wc.median():>9.0f}")
print("="*72)

print("\nChunks per category:")
print(f"  {'Category':<25}", end="")
for name in strategies:
    print(f"  {name:>18}", end="")
print()
print("-"*90)

categories = list(PDF_SOURCES.keys())
for cat in categories:
    print(f"  {cat:<25}", end="")
    for df in strategies.values():
        n = len(df[df["category"] == cat])
        print(f"  {n:>18,}", end="")
    print()
print("-"*90)
print(f"  {'TOTAL':<25}", end="")
for df in strategies.values():
    print(f"  {len(df):>18,}", end="")
print()

print("\nInterpretation:")
print("  S1 Sliding-512  → good general baseline, used in current pipeline")
print("  S2 Sliding-256  → more chunks, better precision for cross-encoder (Week 3)")
print("  S3 Paragraph    → respects document structure, variable length chunks")

## Cell 7 — Save Chunking Stats\n",
    "Best strategy will be selected in Week 3 after re-ranking evaluation."

In [ ]:
stats = {
    "strategies": {
        name: {
            "total_chunks" : len(df),
            "avg_words"    : round(df["word_count"].mean(), 1),
            "min_words"    : int(df["word_count"].min()),
            "max_words"    : int(df["word_count"].max()),
            "by_category"  : df.groupby("category")["chunk_id"].count().to_dict()
        }
        for name, df in strategies.items()
    }
}
with open(CHUNK_STATS, "w") as f:
    json.dump(stats, f, indent=2)
print(f"Saved: pdf_chunking/Results/chunking_stats.json")
print()
print("3 chunk files ready for Week 3 evaluation:")
print(f"  chunks_512w.csv      : {len(df_512w):,} chunks")
print(f"  chunks_256w.csv      : {len(df_256w):,} chunks")
print(f"  chunks_paragraph.csv : {len(df_paragraph):,} chunks")
print()
print("Best strategy → decided in Week 3 after cross-encoder re-ranking evaluation.")

## Cell 8 — Week 2 Complete Summary

In [ ]:
print("="*65)
print("  WEEK 2 COMPLETE — ALL OUTPUTS")
print("="*65)

week2_files = [
    ("Dense Metrics",    "dense_retrieval/Results/dense_metrics.json"),
    ("Dense Results",    "dense_retrieval/Results/minilm_results.csv"),
    ("FAISS Index",      "dense_retrieval/Results/indexes/faiss_minilm.index"),
    ("Hybrid Metrics",   "hybrid/Results/hybrid_metrics.json"),
    ("Hybrid Alpha",     "hybrid/Results/hybrid_alpha0.7_results.csv"),
    ("Chunks 512w",      "Dataset/chunks/chunks_512w.csv"),
    ("Chunks 256w",      "Dataset/chunks/chunks_256w.csv"),
    ("Chunks Paragraph", "Dataset/chunks/chunks_paragraph.csv"),
    ("Chunk Stats",      "pdf_chunking/Results/chunking_stats.json"),
]

for label, rel_path in week2_files:
    full = os.path.join(str(_root), rel_path)
    if os.path.exists(full):
        size = os.path.getsize(full) / 1024
        unit = "KB" if size < 1024 else "MB"
        size = size if size < 1024 else size / 1024
        print(f"  [OK]  {label:<22}  {size:>6.1f} {unit}")
    else:
        print(f"  [--]  {label:<22}  NOT FOUND")

print("="*65)
print()
print("Retrieval Results (FiQA Test):")
print("  BM25 Baseline       NDCG@10 = 0.2169  (Week 1)")
print("  MiniLM-Dense        NDCG@10 = 0.3687  +70.0%")
print("  Hybrid-RRF (k=60)   NDCG@10 = 0.3519  +62.2%")
print("  Hybrid-Alpha (0.7)  NDCG@10 = 0.3791  +74.8%  ◀ BEST")
print()
print("Knowledge Base (3 strategies ready for Week 3):")
print(f"  chunks_512w.csv      : {len(df_512w):,} chunks")
print(f"  chunks_256w.csv      : {len(df_256w):,} chunks")
print(f"  chunks_paragraph.csv : {len(df_paragraph):,} chunks")
print()
print("Next → Week 3: Fine-tuned MiniLM + Cross-Encoder Re-Ranking")